# RQ5, Part 2: Real Bootstrap Confidence Interval

The real, 1,000-resample bootstrap that quantifies honest uncertainty around the cross-domain gap -- the analysis that revealed the point estimate alone overstates confidence (bootstrap 95% CI [0.008, 0.257], only 57.5% of resamples support the equivalence threshold).

**Requires the same real data as Part 1.**

In [1]:
!pip install -q pandas numpy statsmodels scipy || pip install -q pandas numpy statsmodels scipy --break-system-packages

In [3]:
"""
RQ5 Real Bootstrap CI: quantifying uncertainty around the cross-domain
effect-size difference, using real resampling on the real underlying data.
"""
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
N_BOOT = 1000

def rq1_project_f2(df, target):
    df = df.copy()
    df["era_binary"] = (df["era"] == "ai_era").astype(int)
    try:
        full = smf.logit(f"{target} ~ (loc + cyclomatic_complexity + num_functions + num_files_changed) * era_binary", data=df).fit(disp=0)
        reduced = smf.logit(f"{target} ~ loc + cyclomatic_complexity + num_functions + num_files_changed + era_binary", data=df).fit(disp=0)
        r2f = 1 - (full.llf / full.llnull)
        r2r = 1 - (reduced.llf / reduced.llnull)
        return (r2f - r2r) / (1 - r2f)
    except Exception:
        return np.nan

def rq4_f2(df):
    df = df.copy()
    full = smf.ols("remediation_days ~ C(weakness_group) + C(industry_group) + disclosure_year", data=df).fit()
    reduced = smf.ols("remediation_days ~ C(weakness_group)", data=df).fit()
    return (full.rsquared - reduced.rsquared) / (1 - full.rsquared)

def consolidate_weakness(cat):
    cat = str(cat)
    if "Revenue" in cat: return "Revenue Recognition"
    if "ITGC" in cat: return "ITGC"
    if "Complex" in cat or "Warrant" in cat: return "Complex Transactions/Instruments"
    if "Control Environment" in cat or "Staffing" in cat or "Risk Assessment" in cat or "Segregation" in cat: return "Control Environment/Staffing"
    return "Other"

def consolidate_industry(ind):
    ind = str(ind)
    if "Technology" in ind: return "Technology"
    if "Biotech" in ind or "Healthcare" in ind: return "Biotech/Healthcare"
    if "Manufacturing" in ind or "Industrial" in ind or "Aerospace" in ind or "Mining" in ind: return "Manufacturing/Industrial"
    if "SPAC" in ind: return "SPAC"
    if "Energy" in ind: return "Energy"
    if "Financial" in ind or "Insurance" in ind or "Real Estate" in ind: return "Financial/Real Estate"
    return "Media/Consumer/Other"

camel = pd.read_csv("rq1_refined_with_real_issue_type.csv")
kafka = pd.read_csv("kafka_real_mined_dataset.csv")
tika = pd.read_csv("tika_real_mined_dataset.csv")
rq4 = pd.read_csv("rq4_real_sec_edgar_dataset_FINAL.csv")
rq4["weakness_group"] = rq4["weakness_category"].apply(consolidate_weakness)
rq4["industry_group"] = rq4["industry"].apply(consolidate_industry)
rq4["disclosure_year"] = pd.to_datetime(rq4["disclosure_date"]).dt.year

qa_diffs, audit_diffs, gaps = [], [], []
for b in range(N_BOOT):
    c_s = camel.sample(n=len(camel), replace=True, random_state=b)
    k_s = kafka.sample(n=len(kafka), replace=True, random_state=b+10000)
    t_s = tika.sample(n=len(tika), replace=True, random_state=b+20000)
    r4_s = rq4.sample(n=len(rq4), replace=True, random_state=b+30000)

    f2_c = rq1_project_f2(c_s, "defect_prone_strict")
    f2_k = rq1_project_f2(k_s, "defect_prone")
    f2_t = rq1_project_f2(t_s, "defect_prone")
    qa_avg = np.nanmean([f2_c, f2_k, f2_t])

    try:
        f2_audit = rq4_f2(r4_s)
    except Exception:
        continue

    if np.isnan(qa_avg) or np.isnan(f2_audit):
        continue

    qa_diffs.append(qa_avg)
    audit_diffs.append(f2_audit)
    gaps.append(abs(f2_audit - qa_avg))

    if (b+1) % 200 == 0:
        print(f"  {b+1}/{N_BOOT} bootstrap resamples complete")

gaps = np.array(gaps)
print(f"\nReal bootstrap resamples completed: {len(gaps)}")
print(f"QA domain f2: mean={np.mean(qa_diffs):.4f}, 95% CI=[{np.percentile(qa_diffs,2.5):.4f}, {np.percentile(qa_diffs,97.5):.4f}]")
print(f"Audit domain f2: mean={np.mean(audit_diffs):.4f}, 95% CI=[{np.percentile(audit_diffs,2.5):.4f}, {np.percentile(audit_diffs,97.5):.4f}]")
print(f"Gap |audit - QA|: mean={np.mean(gaps):.4f}, 95% CI=[{np.percentile(gaps,2.5):.4f}, {np.percentile(gaps,97.5):.4f}]")
print(f"Proportion of bootstrap gaps below 0.10 threshold: {(gaps < 0.10).mean()*100:.1f}%")


  200/1000 bootstrap resamples complete
  400/1000 bootstrap resamples complete
  600/1000 bootstrap resamples complete
  800/1000 bootstrap resamples complete
  1000/1000 bootstrap resamples complete

Real bootstrap resamples completed: 1000
QA domain f2: mean=0.0163, 95% CI=[0.0069, 0.0315]
Audit domain f2: mean=0.1348, 95% CI=[0.0396, 0.2904]
Gap |audit - QA|: mean=0.1185, 95% CI=[0.0212, 0.2748]
Proportion of bootstrap gaps below 0.10 threshold: 43.9%


## Caveat: comparing f² across model families

The RQ1 QA-domain effect sizes come from **logistic regression** models
(binary defect-prone outcome, N in the thousands per project) via
`1 - (llf / llnull)` pseudo-R², while the RQ4 audit-domain effect size comes
from an **OLS** model (continuous remediation-days outcome, N=113) via
ordinary R². Cohen's f² is computed the same way in both cases
(`(R²_full - R²_reduced) / (1 - R²_full)`), but the underlying R²/pseudo-R²
quantities are not measuring the same thing: OLS R² is the share of outcome
variance explained; McFadden's pseudo-R² (used here for the logistic models)
does not have that direct variance-explained interpretation and is known to
run numerically smaller than OLS R² for comparable model fit.

This means the 0.0534 point-estimate gap (and the bootstrap distribution
built from it) should be read as a **comparison of two Cohen's-f²-scaled
quantities**, not as a comparison of two directly equivalent "percent of
variance explained" figures. The qualitative conclusion below (that the
domains are not clearly equivalent once uncertainty is taken into account)
is still defensible, but the report should state this comparability
limitation explicitly rather than imply the two f² values are on an
identical footing.

## Report-ready interpretation

### What the point estimate alone would suggest (Part 1)

The QA-domain average effect size (f²=0.0091, averaged across Camel/Hadoop,
Kafka, and Tika) and the audit-domain effect size (f²=0.0624, RQ4) differ by
0.0534, which sits below the pre-specified 0.10 equivalence threshold. Taken
at face value, this would suggest the two domains show comparably small
(negligible-to-small) era-related effect sizes.

### Why the point estimate alone is not the honest answer

The 1,000-resample bootstrap shows the gap has a 95% CI of **[0.021, 0.275]**,
with a mean of 0.1185 -- itself above the 0.10 threshold -- and only **43.9%**
of bootstrap resamples fall below the threshold. In other words, more than
half of the resampled estimates of the cross-domain gap exceed the
equivalence threshold, even though the single point estimate from the full
sample happened to fall under it.

### Recommended wording

> **A point-estimate comparison of QA-domain and audit-domain effect sizes
> (Cohen's f² = 0.0091 vs. 0.0624, gap = 0.0534) falls below a pre-specified
> 0.10 equivalence threshold. However, a 1,000-resample bootstrap shows this
> point estimate substantially understates the true uncertainty: the 95%
> bootstrap CI for the gap is [0.021, 0.275], and only 43.9% of resamples
> support the equivalence threshold. The honest conclusion is therefore that
> the cross-domain comparison is inconclusive rather than confirmatory --
> the data are consistent with either a small or a moderate difference
> between domains, and the point estimate should not be reported without
> this uncertainty context.**

### Do not write

- "The QA and audit domains show equivalent effect sizes."
- "RQ5 confirms cross-domain equivalence."
- "The 0.0534 gap proves the domains behave the same way."
- "AI-era effects are equally small across both domains."

These overstate what a single point estimate under substantial bootstrap
uncertainty can support.

### Preferred terminology

Use:

- "point estimate" (not "the result")
- "bootstrap uncertainty" / "bootstrap 95% CI"
- "inconclusive relative to the pre-specified threshold"
- "does not confirm cross-domain equivalence"
- "f² is not directly comparable across model families without caveat"
